In [3]:
import sys

print(sys.executable)

/Users/lizbello/Documents/7mo Semestre/books-music-platform/backend/recommendations-service/.venv/bin/python


In [4]:
import pandas as pd

print(pd.__version__)

3.0.6


In [5]:
# CU-60 — Exploración del dataset de libros

# Exploración inicial de Goodbooks-10k para el desarrollo del modelo de recomendaciones de libros de musa.

In [6]:
import pandas as pd

books = pd.read_csv("../data/raw/books.csv")
ratings = pd.read_csv("../data/raw/ratings.csv")

print("Libros:", books.shape)
print("Ratings:", ratings.shape)

print("BOOKS")
print(books.columns.tolist())

print("\nRATINGS")
print(ratings.columns.tolist())

Libros: (10000, 23)
Ratings: (981756, 3)
BOOKS
['id', 'book_id', 'best_book_id', 'work_id', 'books_count', 'isbn', 'isbn13', 'authors', 'original_publication_year', 'original_title', 'title', 'language_code', 'average_rating', 'ratings_count', 'work_ratings_count', 'work_text_reviews_count', 'ratings_1', 'ratings_2', 'ratings_3', 'ratings_4', 'ratings_5', 'image_url', 'small_image_url']

RATINGS
['book_id', 'user_id', 'rating']


In [7]:
books.head()

,id,book_id,best_book_id,work_id,books_count,isbn,isbn13,authors,original_publication_year,original_title,...,ratings_count,work_ratings_count,work_text_reviews_count,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url
0,1,2767052,2767052,2792775,272,439023483,9.780439e+12,Suzanne Collins,2008.0,The Hunger Games,...,4780653,4942365,155254,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...
1,2,3,3,4640799,491,439554934,9.780440e+12,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,...,4602479,4800065,75867,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...
2,3,41865,41865,3212258,226,316015849,9.780316e+12,Stephenie Meyer,2005.0,Twilight,...,3866839,3916824,95009,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...
3,4,2657,2657,3275794,487,61120081,9.780061e+12,Harper Lee,1960.0,To Kill a Mockingbird,...,3198671,3340896,72586,60427,117415,446835,1001952,1714267,https://images.gr-assets.com/books/1361975680m...,https://images.gr-assets.com/books/1361975680s...
4,5,4671,4671,245494,1356,743273567,9.780743e+12,F. Scott Fitzgerald,1925.0,The Great Gatsby,...,2683664,2773745,51992,86236,197621,606158,936012,947718,https://images.gr-assets.com/books/1490528560m...,https://images.gr-assets.com/books/1490528560s...


In [8]:
ratings["user_id"].nunique()

53424

In [9]:
ratings["book_id"].nunique()

10000

In [10]:
ratings["rating"].value_counts().sort_index()

rating
1     19575
2     63231
3    248623
4    357366
5    292961
Name: count, dtype: int64

In [11]:
ratings.groupby("user_id").size().describe()

count    53424.000000
mean        18.376685
std         26.268690
min          2.000000
25%          3.000000
50%          8.000000
75%         22.000000
max        200.000000
dtype: float64

In [12]:
ratings.groupby("book_id").size().describe()

count    10000.000000
mean        98.175600
std          5.690167
min          8.000000
25%         99.000000
50%        100.000000
75%        100.000000
max        100.000000
dtype: float64

In [13]:
ratings.duplicated(subset=["user_id", "book_id"]).sum()

np.int64(2278)

In [14]:
ratings["rating"].min(), ratings["rating"].max()

(np.int64(1), np.int64(5))

In [15]:
ratings.isnull().sum()

book_id    0
user_id    0
rating     0
dtype: int64

In [16]:
duplicados = ratings[ratings.duplicated(subset=["user_id", "book_id"], keep=False)].sort_values(["user_id", "book_id"])
duplicados.head(20)

,book_id,user_id,rating
459828,4608,7,3
459829,4608,7,2
883645,8946,16,5
883646,8946,16,3
702002,7063,17,5
702003,7063,17,5
613364,6160,25,3
613365,6160,25,4
883648,8946,25,5
883649,8946,25,5


In [17]:
ratings.duplicated().sum()

np.int64(1644)

In [18]:
conflictos = (ratings.groupby(["user_id", "book_id"])["rating"].nunique())
(conflictos > 1).sum()

np.int64(633)

In [19]:
ratings_clean = (ratings.groupby(["user_id", "book_id"], as_index=False)["rating"].mean())
ratings_clean.head()

,user_id,book_id,rating
0,1,1180,4.0
1,1,4893,3.0
2,1,6285,4.0
3,2,8034,4.0
4,2,8855,5.0


In [20]:
print("Rating originales: ", len(ratings))
print("Ratings limpios: ", len(ratings_clean))

print(
    "Duplicados usuario-libro:",
    ratings_clean.duplicated(
        subset=["user_id", "book_id"]
    ).sum()
)

print(
    "Rango:",
    ratings_clean["rating"].min(),
    "-",
    ratings_clean["rating"].max()
)

Rating originales:  981756
Ratings limpios:  979478
Duplicados usuario-libro: 0
Rango: 1.0 - 5.0


In [21]:
interacciones_por_usuario = (
    ratings_clean
    .groupby("user_id")
    .size()
)

In [22]:
usuarios_evaluables = interacciones_por_usuario[
    interacciones_por_usuario >= 2
].index

In [23]:
ratings_evaluables = ratings_clean[
    ratings_clean["user_id"].isin(usuarios_evaluables)
].copy()

In [24]:
ratings_shuffled = ratings_evaluables.sample(
    frac=1,
    random_state=42
)

In [25]:
test = (
    ratings_shuffled
    .groupby("user_id")
    .head(1)
)

In [26]:
train = ratings_evaluables.drop(test.index)

In [27]:
print("Train:", train.shape)
print("Test:", test.shape)

print("Usuarios evaluables:", len(usuarios_evaluables))
print("Usuarios en train:", train["user_id"].nunique())
print("Usuarios en test:", test["user_id"].nunique())

usuarios_test_no_train = (
    set(test["user_id"]) - set(train["user_id"])
)

print(
    "Usuarios de test que no existen en train:",
    len(usuarios_test_no_train)
)

Train: (926054, 3)
Test: (53382, 3)
Usuarios evaluables: 53382
Usuarios en train: 53382
Usuarios en test: 53382
Usuarios de test que no existen en train: 0


In [28]:
books_test_no_train = (set(test["book_id"]) - set(train["book_id"]))

print("Libros en train:", train["book_id"].nunique())
print("Libros en test:", test["book_id"].nunique())

print(
    "Libros de test que no existen en train:",
    len(books_test_no_train)
)

Libros en train: 10000
Libros en test: 9007
Libros de test que no existen en train: 0


In [29]:
import numpy as np

In [30]:
user_ids = train["user_id"].unique()
book_ids = train["book_id"].unique()

user_to_idx = {user_id: idx
               for idx, user_id in enumerate(user_ids)}

book_to_idx = {book_id: idx
               for idx, book_id in enumerate(book_ids)}

In [31]:
train_mf = train.copy()
test_mf = test.copy()

In [32]:
train_mf["user_idx"] = train_mf["user_id"].map(user_to_idx)
train_mf["book_idx"] = train_mf["book_id"].map(book_to_idx)

test_mf["user_idx"] = test_mf["user_id"].map(user_to_idx)
test_mf["book_idx"] = test_mf["book_id"].map(book_to_idx)

In [33]:
train_mf.head()

,user_id,book_id,rating,user_idx,book_idx
1,1,4893,3.0,0,0
2,1,6285,4.0,0,1
3,2,8034,4.0,1,2
5,2,9762,4.0,1,3
6,3,9014,1.0,2,4


In [34]:
print("Usuarios:", len(user_to_idx))
print("Libros:", len(book_to_idx))

print(
    "Nulos en train:",
    train_mf[["user_idx", "book_idx"]].isnull().sum().sum()
)

print(
    "Nulos en test:",
    test_mf[["user_idx", "book_idx"]].isnull().sum().sum()
)


Usuarios: 53382
Libros: 10000
Nulos en train: 0
Nulos en test: 0


In [35]:
n_users = len(user_to_idx)
n_books = len(book_to_idx)

n_factors = 20

In [36]:
rng = np.random.default_rng(42)

P = rng.normal(loc=0, scale=0.1, size=(n_users, n_factors))
Q = rng.normal(loc=0, scale=0.1, size=(n_books, n_factors))

In [37]:
print("P:", P.shape)
print("Q:", Q.shape)

P: (53382, 20)
Q: (10000, 20)


In [38]:
P[0]

array([ 0.03047171, -0.10399841,  0.07504512,  0.09405647, -0.19510352,
       -0.13021795,  0.01278404, -0.03162426, -0.00168012, -0.08530439,
        0.0879398 ,  0.07777919,  0.00660307,  0.11272412,  0.04675093,
       -0.08592925,  0.03687508, -0.09588826,  0.08784503, -0.00499259])

In [39]:
global_mean = train_mf["rating"].mean()

In [40]:
user_bias = np.zeros(n_users)
book_bias = np.zeros(n_books)

In [41]:
print("Media: ", global_mean)
print("Bias de usuario ", user_bias)
print("Bias de libro ", book_bias)

Media:  3.8548482413912506
Bias de usuario  [0. 0. 0. ... 0. 0. 0.]
Bias de libro  [0. 0. 0. ... 0. 0. 0.]


In [42]:
ejemplo = train_mf.iloc[0]

u = int(ejemplo["user_idx"])
i = int(ejemplo["book_idx"])
rating_real = ejemplo["rating"]

print("Usuario:", ejemplo["user_id"])
print("Libro:", ejemplo["book_id"])
print("Rating real:", rating_real)
print("Índice usuario:", u)
print("Índice libro:", i)

Usuario: 1.0
Libro: 4893.0
Rating real: 3.0
Índice usuario: 0
Índice libro: 0


In [43]:
prediccion = (
    global_mean
    + user_bias[u]
    + book_bias[i]
    + np.dot(P[u], Q[i])
)

print("Predicción:", prediccion)
print("Rating real:", rating_real)

Predicción: 3.880811615921244
Rating real: 3.0


In [44]:
error = rating_real - prediccion

print("Error:", error)

Error: -0.8808116159212438


In [45]:
learning_rate = 0.01
regularization = 0.02

user_bias[u] += learning_rate * (
    error - regularization * user_bias[u]
)

book_bias[i] += learning_rate * (
    error - regularization * book_bias[i]
)

In [46]:
p_u_old = P[u].copy()

P[u] += learning_rate * (
    error * Q[i] - regularization * P[u]
)

Q[i] += learning_rate * (
    error * p_u_old - regularization * Q[i]
)


In [47]:
nueva_prediccion = (
    global_mean
    + user_bias[u]
    + book_bias[i]
    + np.dot(P[u], Q[i])
)

nuevo_error = rating_real - nueva_prediccion

print("Antes:")
print("Predicción:", prediccion)
print("Error:", error)

print("\nDespués de una actualización:")
print("Predicción:", nueva_prediccion)
print("Error:", nuevo_error)

Antes:
Predicción: 3.880811615921244
Error: -0.8808116159212438

Después de una actualización:
Predicción: 3.860467940300593
Error: -0.8604679403005928


In [48]:
rng = np.random.default_rng(42)

P = rng.normal(
    loc=0,
    scale=0.1,
    size=(n_users, n_factors)
)

Q = rng.normal(
    loc=0,
    scale=0.1,
    size=(n_books, n_factors)
)

user_bias = np.zeros(n_users)
book_bias = np.zeros(n_books)

global_mean = train_mf["rating"].mean()

In [49]:
train_users = train_mf["user_idx"].to_numpy(dtype=np.int32)
train_books = train_mf["book_idx"].to_numpy(dtype=np.int32)
train_ratings = train_mf["rating"].to_numpy(dtype=np.float32)

In [50]:
print(train_users.shape)
print(train_books.shape)
print(train_ratings.shape)

(926054,)
(926054,)
(926054,)


In [51]:
epochs = 3
learning_rate = 0.01
regularization = 0.02

for epoch in range(epochs):

    indices = rng.permutation(len(train_ratings))
    squared_error = 0.0

    for idx in indices:

        u = train_users[idx]
        i = train_books[idx]
        rating = train_ratings[idx]

        pred = (
            global_mean
            + user_bias[u]
            + book_bias[i]
            + np.dot(P[u], Q[i])
        )

        error = rating - pred
        squared_error += error ** 2

        p_u_old = P[u].copy()

        user_bias[u] += learning_rate * (
            error - regularization * user_bias[u]
        )

        book_bias[i] += learning_rate * (
            error - regularization * book_bias[i]
        )

        P[u] += learning_rate * (
            error * Q[i] - regularization * P[u]
        )

        Q[i] += learning_rate * (
            error * p_u_old - regularization * Q[i]
        )

    rmse_train = np.sqrt(
        squared_error / len(train_ratings)
    )

    print(
        f"Epoch {epoch + 1}/{epochs} - "
        f"RMSE train: {rmse_train:.4f}"
    )

Epoch 1/3 - RMSE train: 0.9205
Epoch 2/3 - RMSE train: 0.8551
Epoch 3/3 - RMSE train: 0.8316


In [52]:
test_users = test_mf["user_idx"].to_numpy(dtype=np.int32)
test_books = test_mf["book_idx"].to_numpy(dtype=np.int32)
test_ratings = test_mf["rating"].to_numpy(dtype=np.float32)

In [53]:
print(test_users.shape)
print(test_books.shape)
print(test_ratings.shape)

(53382,)
(53382,)
(53382,)


In [54]:
test_predictions = (
    global_mean
    + user_bias[test_users]
    + book_bias[test_books]
    + np.sum(
        P[test_users] * Q[test_books],
        axis=1
    )
)

In [55]:
print(test_predictions.shape)

(53382,)


In [56]:
test_errors = test_ratings - test_predictions

In [57]:
rmse_test = np.sqrt(
    np.mean(test_errors ** 2)
)

mae_test = np.mean(
    np.abs(test_errors)
)

print(f"RMSE test: {rmse_test:.4f}")
print(f"MAE test: {mae_test:.4f}")

RMSE test: 0.8956
MAE test: 0.7139


In [58]:
user_id = 1
user_idx = user_to_idx[user_id]

In [59]:
libros_vistos = set(
    train_mf.loc[
        train_mf["user_id"] == user_id,
        "book_id"
    ]
)

print("Libros vistos:", len(libros_vistos))

Libros vistos: 2


In [60]:
libros_candidatos = [
    book_id
    for book_id in book_to_idx
    if book_id not in libros_vistos
]

print("Libros candidatos:", len(libros_candidatos))

Libros candidatos: 9998


In [61]:
candidate_indices = np.array(
    [book_to_idx[book_id] for book_id in libros_candidatos],
    dtype=np.int32
)

In [62]:
scores = (
    global_mean
    + user_bias[user_idx]
    + book_bias[candidate_indices]
    + np.sum(
        P[user_idx] * Q[candidate_indices],
        axis=1
    )
)

In [63]:
top_n = 10

top_positions = np.argsort(scores)[::-1][:top_n]

In [64]:
recomendaciones = []

for position in top_positions:
    recomendaciones.append({
        "book_id": libros_candidatos[position],
        "score": scores[position]
    })

In [65]:
for recomendacion in recomendaciones:
    print(
        "book_id:",
        recomendacion["book_id"],
        "- score:",
        round(recomendacion["score"], 3)
    )

book_id: 3628 - score: 4.765
book_id: 5207 - score: 4.726
book_id: 6590 - score: 4.712
book_id: 862 - score: 4.708
book_id: 6920 - score: 4.674
book_id: 6361 - score: 4.674
book_id: 1308 - score: 4.67
book_id: 4868 - score: 4.657
book_id: 4483 - score: 4.649
book_id: 9566 - score: 4.64


In [67]:
recomendaciones_df = pd.DataFrame(recomendaciones)
recomendaciones_df

,book_id,score
0,3628,4.765259
1,5207,4.725820
2,6590,4.712499
3,862,4.707634
4,6920,4.674473
5,6361,4.673529
6,1308,4.670328
7,4868,4.657322
8,4483,4.648693
9,9566,4.639810


In [69]:
book_info = books[["id", "title", "authors", "average_rating", "image_url"]].copy()
top_recomendaciones = recomendaciones_df.merge(book_info, left_on="book_id", right_on="id", how="left")

top_recomendaciones = top_recomendaciones[["book_id", "title", "authors", "score", "average_rating", "image_url"]]

top_recomendaciones

,book_id,title,authors,score,average_rating,image_url
0,3628,The Complete Calvin and Hobbes,Bill Watterson,4.765259,4.82,https://images.gr-assets.com/books/1473064526m...
1,5207,The Days Are Just Packed: A Calvin and Hobbes ...,Bill Watterson,4.725820,4.68,https://images.gr-assets.com/books/1418211162m...
2,6590,The Authoritative Calvin and Hobbes: A Calvin ...,Bill Watterson,4.712499,4.73,https://images.gr-assets.com/books/1391114055m...
3,862,"Words of Radiance (The Stormlight Archive, #2)",Brandon Sanderson,4.707634,4.77,https://images.gr-assets.com/books/1391535251m...
4,6920,The Indispensable Calvin and Hobbes,Bill Watterson,4.674473,4.73,https://s.gr-assets.com/assets/nophoto/book/11...
5,6361,There's Treasure Everywhere: A Calvin and Hobb...,Bill Watterson,4.673529,4.74,https://s.gr-assets.com/assets/nophoto/book/11...
6,1308,A Court of Mist and Fury (A Court of Thorns an...,Sarah J. Maas,4.670328,4.72,https://images.gr-assets.com/books/1485259138m...
7,4868,Jesus the Christ,James E. Talmage,4.657322,4.63,https://s.gr-assets.com/assets/nophoto/book/11...
8,4483,It's a Magical World: A Calvin and Hobbes Coll...,Bill Watterson,4.648693,4.75,https://images.gr-assets.com/books/1437420710m...
9,9566,Attack of the Deranged Mutant Killer Monster S...,Bill Watterson,4.639810,4.72,https://images.gr-assets.com/books/1384735429m...


In [70]:
# Content-based filtering

In [72]:
book_tags = pd.read_csv("../data/raw/book_tags.csv")
tags = pd.read_csv("../data/raw/tags.csv")

print("book_tags:", book_tags.shape)
print("tags:", tags.shape)

display(book_tags.head())
display(tags.head())

book_tags: (999912, 3)
tags: (34252, 2)


,goodreads_book_id,tag_id,count
0,1,30574,167697
1,1,11305,37174
2,1,11557,34173
3,1,8717,12986
4,1,33114,12716


,tag_id,tag_name
0,0,-
1,1,--1-
2,2,--10-
3,3,--12-
4,4,--122-


In [74]:
print(book_tags.columns)
print(tags.columns)

Index(['goodreads_book_id', 'tag_id', 'count'], dtype='str')
Index(['tag_id', 'tag_name'], dtype='str')


In [76]:
book_tags_named = book_tags.merge(
    tags,
    on="tag_id",
    how="left"
)

book_tags_named.head(20)

,goodreads_book_id,tag_id,count,tag_name
0,1,30574,167697,to-read
1,1,11305,37174,fantasy
2,1,11557,34173,favorites
3,1,8717,12986,currently-reading
4,1,33114,12716,young-adult
5,1,11743,9954,fiction
6,1,14017,7169,harry-potter
7,1,5207,6221,books-i-own
8,1,22743,4974,owned
9,1,32989,4364,ya


In [78]:
tags_populares = (
    book_tags_named
    .groupby("tag_name")["count"]
    .sum()
    .sort_values(ascending=False)
    .head(30)
)

tags_populares

tag_name
to-read               140718761
currently-reading       7507958
favorites               4503173
fiction                 3688819
fantasy                 3548157
young-adult             1848306
classics                1756920
books-i-own             1317235
romance                 1231926
owned                   1224279
ya                       898334
mystery                  872282
non-fiction              857901
historical-fiction       815421
series                   782637
science-fiction          703866
sci-fi                   597325
paranormal               542559
kindle                   506882
contemporary             486001
book-club                446685
library                  429171
favourites               423289
owned-books              417384
horror                   415467
default                  405326
audiobook                388991
urban-fantasy            374689
nonfiction               371049
to-buy                   363276
Name: count, dtype: int64

In [80]:
books[
    ["id", "book_id", "best_book_id", "title"]
].head(10)

,id,book_id,best_book_id,title
0,1,2767052,2767052,"The Hunger Games (The Hunger Games, #1)"
1,2,3,3,Harry Potter and the Sorcerer's Stone (Harry P...
2,3,41865,41865,"Twilight (Twilight, #1)"
3,4,2657,2657,To Kill a Mockingbird
4,5,4671,4671,The Great Gatsby
5,6,11870085,11870085,The Fault in Our Stars
6,7,5907,5907,The Hobbit
7,8,5107,5107,The Catcher in the Rye
8,9,960,960,"Angels & Demons (Robert Langdon, #1)"
9,10,1885,1885,Pride and Prejudice


In [82]:
book_tags[
    ["goodreads_book_id"]
].drop_duplicates().head(10)

,goodreads_book_id
0,1
100,2
200,3
300,5
400,6
500,8
600,10
700,11
800,13
900,21


In [84]:
tag_book_ids = set(book_tags["goodreads_book_id"])

matches_id = books["id"].isin(tag_book_ids).sum()
matches_book_id = books["book_id"].isin(tag_book_ids).sum()
matches_best_book_id = books["best_book_id"].isin(tag_book_ids).sum()

print("Coincidencias con id:", matches_id)
print("Coincidencias con book_id:", matches_book_id)
print("Coincidencias con best_book_id:", matches_best_book_id)

Coincidencias con id: 812
Coincidencias con book_id: 10000
Coincidencias con best_book_id: 9759


In [86]:
book_features = book_tags_named.merge(
    books[
        [
            "id",
            "book_id",
            "title",
            "authors"
        ]
    ],
    left_on="goodreads_book_id",
    right_on="book_id",
    how="inner"
)

book_features.head(20)

,goodreads_book_id,tag_id,count,tag_name,id,book_id,title,authors
0,1,30574,167697,to-read,27,1,Harry Potter and the Half-Blood Prince (Harry ...,"J.K. Rowling, Mary GrandPré"
1,1,11305,37174,fantasy,27,1,Harry Potter and the Half-Blood Prince (Harry ...,"J.K. Rowling, Mary GrandPré"
2,1,11557,34173,favorites,27,1,Harry Potter and the Half-Blood Prince (Harry ...,"J.K. Rowling, Mary GrandPré"
3,1,8717,12986,currently-reading,27,1,Harry Potter and the Half-Blood Prince (Harry ...,"J.K. Rowling, Mary GrandPré"
4,1,33114,12716,young-adult,27,1,Harry Potter and the Half-Blood Prince (Harry ...,"J.K. Rowling, Mary GrandPré"
5,1,11743,9954,fiction,27,1,Harry Potter and the Half-Blood Prince (Harry ...,"J.K. Rowling, Mary GrandPré"
6,1,14017,7169,harry-potter,27,1,Harry Potter and the Half-Blood Prince (Harry ...,"J.K. Rowling, Mary GrandPré"
7,1,5207,6221,books-i-own,27,1,Harry Potter and the Half-Blood Prince (Harry ...,"J.K. Rowling, Mary GrandPré"
8,1,22743,4974,owned,27,1,Harry Potter and the Half-Blood Prince (Harry ...,"J.K. Rowling, Mary GrandPré"
9,1,32989,4364,ya,27,1,Harry Potter and the Half-Blood Prince (Harry ...,"J.K. Rowling, Mary GrandPré"


In [88]:
print("Filas:", len(book_features))
print("Libros únicos:", book_features["id"].nunique())
print("Tags únicos:", book_features["tag_id"].nunique())

Filas: 999912
Libros únicos: 10000
Tags únicos: 34252


In [90]:
tags_no_contenido = {
    "to-read",
    "currently-reading",
    "favorites",
    "favourites",
    "books-i-own",
    "owned",
    "owned-books",
    "re-read",
    "to-buy",
    "library",
    "kindle",
    "audiobook",
    "default"
}

book_features_clean = book_features[
    ~book_features["tag_name"].isin(tags_no_contenido)
].copy()

print("Filas antes:", len(book_features))
print("Filas después:", len(book_features_clean))
print("Libros:", book_features_clean["id"].nunique())
print("Tags:", book_features_clean["tag_name"].nunique())

Filas antes: 999912
Filas después: 889165
Libros: 10000
Tags: 34239


In [92]:
ejemplo_libro = (
    book_features_clean[
        book_features_clean["goodreads_book_id"] == 1
    ]
    .sort_values("count", ascending=False)
    [
        ["title", "tag_name", "count"]
    ]
    .head(20)
)

ejemplo_libro

,title,tag_name,count
1,Harry Potter and the Half-Blood Prince (Harry ...,fantasy,37174
4,Harry Potter and the Half-Blood Prince (Harry ...,young-adult,12716
5,Harry Potter and the Half-Blood Prince (Harry ...,fiction,9954
6,Harry Potter and the Half-Blood Prince (Harry ...,harry-potter,7169
9,Harry Potter and the Half-Blood Prince (Harry ...,ya,4364
10,Harry Potter and the Half-Blood Prince (Harry ...,series,3857
12,Harry Potter and the Half-Blood Prince (Harry ...,magic,3374
13,Harry Potter and the Half-Blood Prince (Harry ...,childrens,2408
16,Harry Potter and the Half-Blood Prince (Harry ...,adventure,1742
17,Harry Potter and the Half-Blood Prince (Harry ...,children,1560


In [95]:
frecuencia_tags = (
    book_features_clean
    .groupby("tag_name")["id"]
    .nunique()
    .sort_values(ascending=False)
)

frecuencia_tags.describe()

count    34239.000000
mean        25.969158
std        220.038432
min          1.000000
25%          1.000000
50%          1.000000
75%          5.000000
max       9097.000000
Name: id, dtype: float64

In [97]:
print("Tags que aparecen en 1 solo libro:",
      (frecuencia_tags == 1).sum())

print("Tags que aparecen en menos de 5 libros:",
      (frecuencia_tags < 5).sum())

print("Tags que aparecen en menos de 10 libros:",
      (frecuencia_tags < 10).sum())

print("Tags que aparecen en 10 o más libros:",
      (frecuencia_tags >= 10).sum())

Tags que aparecen en 1 solo libro: 17158
Tags que aparecen en menos de 5 libros: 25267
Tags que aparecen en menos de 10 libros: 28749
Tags que aparecen en 10 o más libros: 5490


In [99]:
tags_validos = frecuencia_tags[
    frecuencia_tags >= 10
].index

book_features_filtered = book_features_clean[
    book_features_clean["tag_name"].isin(tags_validos)
].copy()

print("Filas:", len(book_features_filtered))
print("Libros:", book_features_filtered["id"].nunique())
print("Tags:", book_features_filtered["tag_name"].nunique())

Filas: 827912
Libros: 10000
Tags: 5490


In [106]:
from scipy.sparse import csr_matrix

In [107]:
content_book_ids = sorted(
    book_features_filtered["id"].unique()
)

content_tags = sorted(
    book_features_filtered["tag_name"].unique()
)

content_book_to_idx = {
    book_id: idx
    for idx, book_id in enumerate(content_book_ids)
}

content_tag_to_idx = {
    tag: idx
    for idx, tag in enumerate(content_tags)
}

In [109]:
content_data = book_features_filtered.copy()

content_data["book_idx"] = (
    content_data["id"].map(content_book_to_idx)
)

content_data["tag_idx"] = (
    content_data["tag_name"].map(content_tag_to_idx)
)

In [111]:
book_tag_matrix = csr_matrix(
    (
        content_data["count"].astype(float),
        (
            content_data["book_idx"],
            content_data["tag_idx"]
        )
    ),
    shape=(
        len(content_book_ids),
        len(content_tags)
    )
)

print("Shape:", book_tag_matrix.shape)
print("Valores almacenados:", book_tag_matrix.nnz)

Shape: (10000, 5490)
Valores almacenados: 827906


In [112]:
from sklearn.feature_extraction.text import TfidfTransformer

tfidf = TfidfTransformer()

book_tfidf_matrix = tfidf.fit_transform(
    book_tag_matrix
)

print("Shape TF-IDF:", book_tfidf_matrix.shape)
print("Valores TF-IDF:", book_tfidf_matrix.nnz)

Shape TF-IDF: (10000, 5490)
Valores TF-IDF: 827906
